In [2]:
import os
import fnmatch

def glob_with_depth(base_path, pattern="*", max_depth=None):
    matches = []
    base_depth = base_path.rstrip(os.sep).count(os.sep)

    for root, dirs, files in os.walk(base_path):
        current_depth = root.count(os.sep) - base_depth
        if max_depth is not None and current_depth > max_depth:
            dirs[:] = []
            continue
        for filename in files:
            if fnmatch.fnmatch(filename, pattern):
                matches.append(os.path.join(root, filename))
    return matches


base_dir = "archive (12)"

all_paired_data = []
summary = []

participant_ids = [p for p in range(1, 20) if p != 11]  # P1–P19, skip P11

for pid in participant_ids:
    participant = f"P{pid}"
    data_path = os.path.join(base_dir, participant, "C", "Data")

    if not os.path.exists(data_path):
        print(f"[SKIP] {participant}: path not found → {data_path}")
        continue

    # Find all center annotation files
    annotation_files = glob_with_depth(data_path, "*_center.txt", max_depth=2)

    # Pair each annotation with its image
    paired = []
    missing = []

    for ann_path in annotation_files:
        img_path = ann_path.rsplit("_center.txt", 1)[0] + ".tiff.png"
        if os.path.exists(img_path):
            paired.append((img_path, ann_path))
        else:
            missing.append(img_path)

    all_paired_data.extend(paired)
    summary.append({
        "participant": participant,
        "valid_pairs": len(paired),
        "missing_images": len(missing),
        "total_annotations": len(annotation_files)
    })

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'Participant':<14} {'Valid Pairs':>11} {'Missing':>8} {'Annotations':>12}")
print("-" * 50)
for s in summary:
    print(f"{s['participant']:<14} {s['valid_pairs']:>11} {s['missing_images']:>8} {s['total_annotations']:>12}")

print("-" * 50)
print(f"{'TOTAL':<14} {len(all_paired_data):>11}")

# all_paired_data → list of (img_path, ann_path) tuples across all participants


Participant    Valid Pairs  Missing  Annotations
--------------------------------------------------
P1                     300        0          300
P2                     300        0          300
P3                     300        0          300
P4                     300        0          300
P5                     300        0          300
P6                     300        0          300
P7                     300        0          300
P8                     250        0          250
P9                       0        0            0
P10                    300        0          300
P12                    300        0          300
P13                    300        0          300
P14                    300        0          300
P15                    300        0          300
P16                    300        0          300
P17                    300        0          300
P18                    300        0          300
P19                    300        0          300
-----------------

In [ ]:
# ── Annotation quality filter ─────────────────────────────────────────────────
# Outlier analysis revealed samples where GT = (0.000, 0.000) or near-zero on
# both axes. These are failed annotations (the labelling tool defaulted to the
# image origin when it could not detect the pupil). A real pupil center is never
# at the absolute top-left corner of a cropped eye image.
# Threshold: reject any sample where BOTH x < 0.02 AND y < 0.02.

def _has_valid_annotation(ann_path: str, min_val: float = 0.02) -> bool:
    raw = open(ann_path).read().strip().split(",")
    x, y = float(raw[0]), float(raw[1])
    return not (x < min_val and y < min_val)

before = len(all_paired_data)
all_paired_data_clean = [
    (img, ann) for img, ann in all_paired_data
    if _has_valid_annotation(ann)
]
removed = before - len(all_paired_data_clean)

print(f"Total samples     : {before}")
print(f"Bad annotations   : {removed}  (GT stuck at origin — filtered out)")
print(f"Clean dataset     : {len(all_paired_data_clean)}")


In [ ]:
import importlib
import dataset, model, train, eval, utils

for mod in [utils, dataset, model, train, eval]:
    importlib.reload(mod)

from dataset import get_dataloaders
from model   import GazeEstimationModel
from train   import train as run_train
from eval    import evaluate, visualize_predictions, visualize_error_distribution, visualize_spatial_errors

# ── DataLoaders — use the cleaned dataset (bad annotations removed) ────────────
train_loader, val_loader, test_loader = get_dataloaders(
    all_paired_data_clean,          # <-- filtered, no (0,0) annotations
    batch_size=8,
    num_workers=0,
    train_ratio=0.70,
    val_ratio=0.15,
)

# ── Model ──────────────────────────────────────────────────────────────────────
gaze_model = GazeEstimationModel(pretrained=True, dropout_rate=0.4)

# ── Train ──────────────────────────────────────────────────────────────────────
history = run_train(
    gaze_model, train_loader, val_loader,
    num_epochs              = 100,
    learning_rate           = 5e-4,
    weight_decay            = 3e-4,
    grad_accum_steps        = 4,
    early_stopping_patience = 20,
    scheduler_patience      = 6,
    x_loss_weight           = 2.0,
    huber_delta             = 0.05,
    checkpoint_path         = "checkpoints/best_model_v2.pth",
)

# ── Evaluate ───────────────────────────────────────────────────────────────────
metrics = evaluate(gaze_model, test_loader,
                   checkpoint_path="checkpoints/best_model_v2.pth")

# ── Visualize ──────────────────────────────────────────────────────────────────
visualize_predictions(gaze_model, test_loader)
visualize_error_distribution(gaze_model, test_loader)
visualize_spatial_errors(gaze_model, test_loader)


In [1]:

# ── Step 1 test: EyeDetector on a single camera frame ─────────────────────────
import importlib, sys
import eye_detector
importlib.reload(eye_detector)
from eye_detector import EyeDetector

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from dataset import _eval_transform
from model   import GazeEstimationModel
from utils   import load_checkpoint

CHECKPOINT = "checkpoints/best_model_v2.pth"
device     = torch.device("cpu")

gaze_model = GazeEstimationModel(pretrained=False)
load_checkpoint(CHECKPOINT, gaze_model, device=device)
gaze_model.eval()
transform = _eval_transform()

detector = EyeDetector(padding=0.30)
cap      = cv2.VideoCapture(0)
assert cap.isOpened(), "Cannot open camera — check it's connected and not in use."

for _ in range(5):
    cap.read()
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read a frame from the camera."

result = detector.detect(frame)
detector.release()

eye_img   = result["right_eye"] or result["left_eye"]
eye_label = "right" if result["right_eye"] else "left"
pred = None

if eye_img is not None:
    with torch.no_grad():
        inp  = transform(eye_img).unsqueeze(0).to(device)
        pred = gaze_model(inp).squeeze().tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("EyeDetector — single frame test", fontsize=13)

axes[0].imshow(cv2.cvtColor(result["annotated_frame"], cv2.COLOR_BGR2RGB))
axes[0].set_title("Annotated frame"); axes[0].axis("off")

if result["right_eye"]:
    title = "Right eye"
    if pred:
        title += f"\npupil  x={pred[0]:.3f}  y={pred[1]:.3f}"
    axes[1].imshow(np.array(result["right_eye"]))
    axes[1].set_title(title)
else:
    axes[1].set_title("Right eye — not detected")
axes[1].axis("off")

if result["left_eye"]:
    axes[2].imshow(np.array(result["left_eye"]))
    axes[2].set_title("Left eye")
else:
    axes[2].set_title("Left eye — not detected")
axes[2].axis("off")

plt.tight_layout(); plt.show()

print(f"Face detected : {result['face_detected']}")
print(f"Eye used      : {eye_label}")
if pred:
    print(f"Pupil pred    : x={pred[0]:.4f}  y={pred[1]:.4f}  (normalised [0,1])")
else:
    print("No eye detected — point camera at your face and retry.")


ModuleNotFoundError: No module named 'mediapipe.solutions'

In [ ]:

# ── Step 2 test: GazeMapper with synthetic calibration data ───────────────────
import importlib, gaze_mapper
importlib.reload(gaze_mapper)
from gaze_mapper import GazeMapper, build_calibration_points, GRID_ITEMS

import numpy as np
import matplotlib.pyplot as plt

# Simulate a 1920×1080 screen (same maths regardless of actual resolution)
SW, SH = 1920, 1080
cal_screen_pts = build_calibration_points(SW, SH)
print("Calibration screen points (9 grid-cell centres):")
for i, (sx, sy) in enumerate(cal_screen_pts):
    print(f"  [{i}] {GRID_ITEMS[i]:<8}  screen=({sx:4d}, {sy:4d})")

# ── Synthetic pupil data ───────────────────────────────────────────────────────
# Real pupils won't be this perfectly ordered, but this lets us verify the math.
# We simulate: centre of screen (0.5,0.5) maps to centre calibration point,
# edges shift proportionally.  Add a little Gaussian noise.
rng = np.random.default_rng(42)

def _synthetic_pupil(sx, sy, sw, sh, noise=0.01):
    """Fake a pupil position that linearly tracks screen position."""
    px = sx / sw
    py = sy / sh
    px += rng.normal(0, noise)
    py += rng.normal(0, noise)
    return float(np.clip(px, 0, 1)), float(np.clip(py, 0, 1))

cal_pupil_pts = [_synthetic_pupil(sx, sy, SW, SH) for sx, sy in cal_screen_pts]

print("\nSynthetic pupil points:")
for i, (px, py) in enumerate(cal_pupil_pts):
    print(f"  [{i}] {GRID_ITEMS[i]:<8}  pupil=({px:.4f}, {py:.4f})")

# ── Fit the mapper ─────────────────────────────────────────────────────────────
mapper = GazeMapper()
mapper.fit(cal_pupil_pts, cal_screen_pts)

# ── Evaluate: predict each calibration point back ─────────────────────────────
print("\nCalibration round-trip error:")
errors = []
for i, ((px, py), (sx, sy)) in enumerate(zip(cal_pupil_pts, cal_screen_pts)):
    pred_sx, pred_sy = mapper.predict(px, py)
    err = np.sqrt((pred_sx - sx)**2 + (pred_sy - sy)**2)
    errors.append(err)
    print(f"  [{i}] {GRID_ITEMS[i]:<8}  GT=({sx:4d},{sy:4d})  "
          f"pred=({pred_sx:6.1f},{pred_sy:6.1f})  err={err:.1f} px")
print(f"\n  Mean error : {np.mean(errors):.1f} px")
print(f"  Max  error : {np.max(errors):.1f} px")

# ── Test screen_to_grid_cell ───────────────────────────────────────────────────
print("\nscreen_to_grid_cell sanity check:")
test_pts = [(960, 540, "centre"), (0, 0, "top-left corner"),
            (1919, 1079, "bottom-right corner")]
for x, y, label in test_pts:
    cell = mapper.screen_to_grid_cell(x, y, SW, SH)
    item = GRID_ITEMS[cell] if cell is not None else "outside"
    print(f"  ({x:4d},{y:4d}) {label:<25} → cell {cell}  ({item})")

# ── Visualisation ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GazeMapper — synthetic calibration test", fontsize=13)

# Left: pupil space
ax = axes[0]
px_vals = [p[0] for p in cal_pupil_pts]
py_vals = [p[1] for p in cal_pupil_pts]
ax.scatter(px_vals, py_vals, s=80, c="steelblue", zorder=3)
for i, (px, py) in enumerate(cal_pupil_pts):
    ax.annotate(GRID_ITEMS[i], (px, py), textcoords="offset points",
                xytext=(5, 4), fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(1, 0)   # y-flipped to match screen coords
ax.set_xlabel("pupil x"); ax.set_ylabel("pupil y")
ax.set_title("Pupil space (normalised)")
ax.grid(True, alpha=0.3)

# Right: screen space — GT vs predicted
ax = axes[1]
gt_x   = [p[0] for p in cal_screen_pts]
gt_y   = [p[1] for p in cal_screen_pts]
pred_x = [mapper.predict(px, py)[0] for px, py in cal_pupil_pts]
pred_y = [mapper.predict(px, py)[1] for px, py in cal_pupil_pts]

ax.scatter(gt_x,   gt_y,   s=80, c="lime",   label="Ground truth", zorder=4)
ax.scatter(pred_x, pred_y, s=40, c="tomato", marker="x", linewidths=2,
           label="Predicted",    zorder=5)
for i in range(len(cal_screen_pts)):
    ax.plot([gt_x[i], pred_x[i]], [gt_y[i], pred_y[i]],
            "gray", lw=0.8, alpha=0.6)
    ax.annotate(GRID_ITEMS[i], (gt_x[i], gt_y[i]), textcoords="offset points",
                xytext=(5, 4), fontsize=8)

ax.set_xlim(0, SW); ax.set_ylim(SH, 0)
ax.set_xlabel("screen x (px)"); ax.set_ylabel("screen y (px)")
ax.set_title("Screen space — GT vs predicted")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nGazeMapper test PASSED — mapper is working correctly.")


In [ ]:

# ── Step 3: Full live inference — camera → eye detect → gaze model ─────────────
import importlib, eye_detector, gaze_mapper
importlib.reload(eye_detector); importlib.reload(gaze_mapper)
from eye_detector import EyeDetector
from gaze_mapper  import GazeMapper, build_calibration_points, GRID_ITEMS

import cv2, torch
import numpy as np
import matplotlib.pyplot as plt
from dataset import _eval_transform
from model   import GazeEstimationModel
from utils   import load_checkpoint

CHECKPOINT = "checkpoints/best_model_v2.pth"
N_FRAMES   = 60
device     = torch.device("cpu")

gaze_model = GazeEstimationModel(pretrained=False)
load_checkpoint(CHECKPOINT, gaze_model, device=device)
gaze_model.eval()
transform = _eval_transform()
detector  = EyeDetector(padding=0.30)

cap = cv2.VideoCapture(0)
assert cap.isOpened(), "Camera not available."
for _ in range(5): cap.read()

preds_x, preds_y, frames_with_face = [], [], []

print(f"Capturing {N_FRAMES} frames …")
for i in range(N_FRAMES):
    ret, frame = cap.read()
    if not ret:
        continue

    result  = detector.detect(frame)
    eye_img = result["right_eye"] or result["left_eye"]
    label   = "R" if result["right_eye"] else ("L" if result["left_eye"] else "-")

    if eye_img is not None:
        with torch.no_grad():
            inp  = transform(eye_img).unsqueeze(0).to(device)
            pred = gaze_model(inp).squeeze().tolist()

        preds_x.append(pred[0])
        preds_y.append(pred[1])
        frames_with_face.append(i)
        print(f"  [frame {i:03d}]  eye={label}  pupil x={pred[0]:.4f}  y={pred[1]:.4f}")
    else:
        print(f"  [frame {i:03d}]  no face / eye detected")

cap.release()
detector.release()

if not preds_x:
    print("\nNo predictions — point camera at your face and re-run.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Live inference — {len(preds_x)}/{N_FRAMES} frames detected", fontsize=13)

    ax = axes[0]
    sc = ax.scatter(preds_x, preds_y, c=range(len(preds_x)), cmap="plasma", s=40, zorder=3)
    ax.plot(preds_x, preds_y, "gray", lw=0.6, alpha=0.5)
    plt.colorbar(sc, ax=ax, label="frame index")
    ax.set_xlim(0, 1); ax.set_ylim(1, 0)
    ax.set_xlabel("pupil x"); ax.set_ylabel("pupil y")
    ax.set_title("Gaze trajectory (pupil space)"); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(frames_with_face, preds_x, label="pupil x", color="steelblue")
    ax.plot(frames_with_face, preds_y, label="pupil y", color="tomato")
    ax.set_xlabel("frame index"); ax.set_ylabel("normalised position")
    ax.set_ylim(-0.05, 1.05); ax.set_title("Pupil x / y over time")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    print(f"\nSummary over {len(preds_x)} detected frames:")
    print(f"  pupil x  mean={np.mean(preds_x):.4f}  std={np.std(preds_x):.4f}")
    print(f"  pupil y  mean={np.mean(preds_y):.4f}  std={np.std(preds_y):.4f}")
    print("\nFull pipeline working correctly.")


In [ ]:

# ── Step 4 test: TTS engine ────────────────────────────────────────────────────
import importlib, tts_engine
importlib.reload(tts_engine)
from tts_engine import speak, list_voices, set_rate
import time

# Show available system voices
voices = list_voices()
print(f"Found {len(voices)} voice(s):")
for i, v in enumerate(voices):
    print(f"  [{i}] {v.name}")

# Speak every grid item one by one so you can check audio + speed
from gaze_mapper import GRID_ITEMS
set_rate(145)

print(f"\nSpeaking all 9 grid items …")
for item in GRID_ITEMS:
    print(f"  → {item}")
    speak(item, block=True)
    time.sleep(0.25)

print("\nTTS engine OK.")


In [ ]:

# ── Step 3 test: Grid layout preview (matplotlib, no pygame needed) ─────────────
import importlib, grid_ui, gaze_mapper
importlib.reload(grid_ui); importlib.reload(gaze_mapper)
from gaze_mapper import GRID_ITEMS, build_calibration_points

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Simulated screen resolution for preview
SW, SH   = 1920, 1080
MX, MY   = 100, 80        # margins

gw = (SW - 2 * MX) / 3
gh = (SH - 2 * MY) / 3

fig, ax = plt.subplots(figsize=(14, 8), facecolor="#0c0e16")
ax.set_facecolor("#0c0e16")
ax.set_xlim(0, SW); ax.set_ylim(SH, 0)   # y flipped (screen coords)
ax.set_aspect("equal")
ax.axis("off")
fig.suptitle("Grid UI — layout preview (1920×1080)", color="white", fontsize=13)

cal_pts = build_calibration_points(SW, SH, MX, MY)

for i, (label, (cx, cy)) in enumerate(zip(GRID_ITEMS, cal_pts)):
    row, col = divmod(i, 3)
    x = MX + col * gw
    y = MY + row * gh

    # Cell background
    rect = mpatches.FancyBboxPatch(
        (x + 4, y + 4), gw - 8, gh - 8,
        boxstyle="round,pad=0",
        linewidth=1.5, edgecolor="#3c4160", facecolor="#1c2134",
    )
    ax.add_patch(rect)

    # Label
    ax.text(cx, cy, label, color="white", fontsize=22, fontweight="bold",
            ha="center", va="center")

    # Calibration dot (centre of cell)
    ax.plot(cx, cy, "o", color="#00c8ff", markersize=6, zorder=5)

    # Cell index label (small, top-left of cell)
    ax.text(x + 12, y + 22, str(i), color="#555977", fontsize=9)

# Dwell arc example on cell 4 (TOILET — centre)
cx4, cy4 = cal_pts[4]
theta = np.linspace(-np.pi / 2, -np.pi / 2 + 0.75 * 2 * np.pi, 200)
r = 52
ax.plot(cx4 + r * np.cos(theta), cy4 + r * np.sin(theta),
        color="#00c8ff", linewidth=4, label="Dwell arc (75%)")

# Simulated gaze dot
ax.plot(cx4 + 15, cy4 - 10, "o", color="#ff4646", markersize=10, zorder=6,
        label="Gaze dot")
ax.plot(cx4 + 15, cy4 - 10, "o", color="white",   markersize=10, zorder=5,
        markerfacecolor="none", markeredgewidth=1.5)

ax.legend(loc="lower right", facecolor="#1a1d2e", labelcolor="white",
          fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

# Verify grid math
print("Cell-centre → screen_to_grid_cell round-trip:")
from gaze_mapper import GazeMapper
m = GazeMapper()
for i, (cx, cy) in enumerate(cal_pts):
    cell = m.screen_to_grid_cell(cx, cy, SW, SH, MX, MY)
    status = "OK" if cell == i else f"FAIL (got {cell})"
    print(f"  [{i}] {GRID_ITEMS[i]:<8}  centre=({cx:4d},{cy:4d})  → cell {cell}  {status}")

print("\nGrid layout test PASSED. Ready to run the full system.")
